[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Favioleiva/gpubma/blob/main/examples/BFG_Canonical_p30_Shell_Recovery.ipynb)

# Post-BFG Stratified Random Shell Recovery: Canonical p30 Benchmark

This notebook demonstrates the complete validated post-BFG reticular recovery workflow on the canonical p=30 synthetic benchmark (n=2,000, p=30, w1/w2 always-in controls, g=2,000, beta-binomial(1,1)).

### Two complementary components:
1. **Targeted BFG Discovery**: A budgeted search (B=5,000) that isolates high-evidence models, finding the exact global MAP ($x_1-x_{14}, x_{30}$), true causal model ($x_1-x_{15}$), 10/10 exact top-10 models, and 98/100 exact top-100 models.
2. **Representative Uniform Random Shell Recovery**: Simple random sampling without replacement from the undiscovered remainder in each shell to reconstruct reticular score distributions.

### Critical scientific distinction:
- **EXACT FULL ENUMERATION**: All 1,073,741,824 models evaluated; establishes exact global Z, exact global PMP, and exact global PIP.
- **BFG DISCOVERY**: Targeted evaluated set (5,000 models); establishes champions, modes, and genealogy.
- **RANDOM SHELL RECOVERY**: Representative reticular score distributions. Does NOT claim exact global Z, PMP, or PIP.

**Critical Rule:** Targeted BFG discoveries must NOT be pooled equally as if they were random draws when estimating reticular score distributions.


## 1. Environment bootstrap and repository-relative setup

Resolves repository root dynamically, compatible with local development and Google Colab environments without hardcoded absolute paths.


In [ ]:
from pathlib import Path
import json, math, time
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image

roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / "benchmark/exact_p30/reference/REFERENCE_MANIFEST.json").is_file()), None)
if ROOT is None:
    ROOT = Path.cwd()

DATA_PATH = ROOT / "data/synthetic/panel_30_center15.parquet"
SHELL_RECOVERY_DIR = ROOT / "benchmark/shell_recovery"
EXACT_REF_DIR = ROOT / "benchmark/exact_p30/reference"
OUTPUT_DIR = ROOT / "outputs/shell_recovery"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import gpubma
from gpubma.bfg import shell_recovery as sr
from gpubma.bfg.example_workflow import load_example_data, fit_example
print(f"gpubma version: {gpubma.__version__}")
print("Repository root resolved successfully.")


## 2. Load canonical p30 dataset and execute targeted BFG discovery

We run BFG with budget B=5,000 to discover high-evidence models and record the search footprint.


In [ ]:
y, X, controls = load_example_data(
    DATA_PATH,
    "y",
    [f"x{i}" for i in range(1, 31)],
    ["w1", "w2"],
)
print(f"Dataset loaded: n={len(y)}, p={X.shape[1]}, controls={controls.shape[1]}")

t0 = time.perf_counter()
bfg_result, bfg_metrics = fit_example(y, X, controls, budget=5000, seed=12345, device="auto")
discovery_s = time.perf_counter() - t0

print(f"BFG discovery completed in {discovery_s:.3f} s")
print(f"Unique models evaluated: {bfg_result.n_models_evaluated:,} (Hard budget: 5,000)")
best = bfg_result.best_model
print(f"Best model found: ID {best['model_id']} (size k={best['model_size']}, log score={best['log_score']:.4f})")
print(f"Variables: {best['variable_names']}")


## 3. BFG Search Footprint: Mode Discovery vs. Reticular Volume

BFG evaluates 5,000 models out of 2^30 = 1,073,741,824 (0.000466%).
It finds the exact global MAP and generating model, capturing 10/10 exact top-10 models and 98/100 top-100 models.
However, BFG discoveries are concentrated around high-evidence modes. Remainder shells contain only 8-17 observations, requiring stratified random recovery to reconstruct reticular distributions.


In [ ]:
shell_counts = pd.read_csv(SHELL_RECOVERY_DIR / "POST_BFG_RANDOM_SHELL_COUNTS.csv")
display(shell_counts[["k", "N_k", "D_k", "n_k_random", "sample_fraction", "binding_rule"]].head(16))


## 4. Post-BFG Stratified Remainder Sampling

For each candidate shell (size k):
1. Shell population N_k = C(p, k).
2. Exclude all D_k models already discovered by BFG.
3. Draw a uniform simple random sample without replacement of size n_k = min(ceil(0.01 * N_k), cap, N_k - D_k) from the undiscovered remainder.
4. Default recommended cap is **100,000 per shell**.


In [ ]:
# Demonstrate RemainderSampler on central shell k=10
k_demo = 10
N_10 = sr.shell_population_size(30, k_demo)
n_100k = sr.sample_size(30, k_demo, discovered=0, cap=100_000)
print(f"Shell k={k_demo}: N={N_10:,}, 1% target={(N_10 + 99)//100:,}, Sample size at Cap 100k={n_100k:,}")

sampler = sr.RemainderSampler(30, k_demo, seed=2026091401)
demo_ranks = sampler.extend(10)
print(f"First 10 sampled ranks in shell {k_demo}: {demo_ranks}")
for r in demo_ranks[:3]:
    indices = sr.unrank_combination(int(r), 30, k_demo)
    mask = sr.mask_from_rank(int(r), 30, k_demo)
    print(f"  Rank {r:9d} -> active predictors: {indices} -> mask: {mask}")


## 5. Reconstructing Reticular Distributions Without Equal Pooling

**Critical rule:** Targeted BFG discoveries must NOT be pooled equally with random draws.
For candidate shell (size k):
- Discovered models receive weight 1 / N_k each.
- Random remainder models receive weight (N_k - D_k) / (N_k * n_k) each.
Total shell weight sums to 1.0.


In [ ]:
demo_disc = np.array([1000.5, 998.2])  # 2 discovered models
demo_samp = np.array([850.1, 845.3, 860.0])  # 3 remainder samples
values, weights = sr.reconstruct_shell_mixture(demo_disc, demo_samp, N_k=1000)
print("Mixture values:", values)
print("Mixture weights:", weights)
print(f"Weight sum: {weights.sum():.6f}")
print(f"Discovered weight sum: {weights[:2].sum():.4f} (D/N = 2/1000 = 0.002)")
print(f"Sample weight sum: {weights[2:].sum():.4f} ((N-D)/N = 998/1000 = 0.998)")


## 6. Validated Recovery Ladder and Error Analysis Against Frozen Exact Universe

We evaluate the precision/runtime trade-off across three caps against the frozen exact reference (2^30 models):
- **Cap 10,000**: 185,360 random models; recovery runtime 0.527 s; central mean KS = 0.010310; central mean W1 = 1.777813
- **Cap 100,000 (Recommended Default)**: 1,473,136 random models; recovery runtime 2.890 s; central mean KS = 0.003381; central mean W1 = 0.552063
- **Cap 1,000,000**: 8,882,594 random models; recovery runtime 22.489 s; central mean KS = 0.000705; central mean W1 = 0.133140


In [ ]:
cap_summary = pd.read_csv(SHELL_RECOVERY_DIR / "RANDOM_SHELL_CAP_SENSITIVITY.csv")
display(cap_summary[[
    "cap", "random_evaluations", "recovery_runtime_s",
    "central_KS_mean", "central_W1_mean", "KS_median", "W1_median"
]])


## 7. Comparative Timing and Evaluation Accounting Table

Benchmark table comparing full enumeration, BFG discovery only, and BFG combined with stratified shell recovery.


In [ ]:
timing_comp = pd.read_csv(SHELL_RECOVERY_DIR / "BFG_RANDOM_RECOVERY_TIMING_COMPARISON.csv")
display(timing_comp[[
    "Method", "Discovery_evaluations", "Random_recovery_evaluations", "Total_evaluations",
    "Runtime_s", "Models_per_s", "MAP_found", "Top10_overlap", "Top100_overlap", "Reticular_distribution_quality"
]])


## 8. Shell-Recovery Publication Figures

All figures are generated with zero embedded titles and zero embedded captions.
Titles and notes are provided as external Markdown.


In [ ]:
import sys
sys.path.insert(0, str(ROOT / "examples"))
from shell_recovery_figures import render_shell_recovery_figures

catalog = render_shell_recovery_figures(SHELL_RECOVERY_DIR, EXACT_REF_DIR, OUTPUT_DIR)
print(f"Rendered {len(catalog)} title-free figures into outputs/shell_recovery")


### Population-Weighted Post-BFG Shell Recovery


In [ ]:
display(Image(filename=str(OUTPUT_DIR / "post_bfg_random_reticular_ridgeline.png")))


*Note.* Cap=1,000,000 with the unchanged 1% rule. Known BFG models have weight 1/N; random-remainder models have weight (N-D)/(N*n). Both components reconstruct the full shell; targeted discoveries are not treated as random. Gaussian smoothing sigma=3 bins and independent unit-peak normalization are display-only, on the frozen 2,000-bin grid. These are score densities, not posterior mass. Reference support endpoints bound rendering; no support recovery is claimed. Right labels give new random samples; the exact wings are censuses. k=3 and k=27 retain only 41 samples. All shell champions were already found by BFG and appear as overlays.


### Post-BFG Reconstruction Against Frozen Exact Shells


In [ ]:
display(Image(filename=str(OUTPUT_DIR / "post_bfg_random_vs_exact_reticular_ridgeline.png")))


*Note.* Post-BFG reconstructed reticular score distributions (teal) overlaid directly on the frozen exact full-enumeration distributions (grey). The close agreement across all 31 shells confirms high fidelity of stratified remainder sampling and mixture weighting.


### Reconstruction Error by Shell and Cap


In [ ]:
display(Image(filename=str(OUTPUT_DIR / "random_shell_error_by_k.png")))


*Note.* Exact weighted-CDF Kolmogorov-Smirnov distance (percentage points) and Wasserstein-1 distance (score units) across model size k for 10k, 100k, and 1M caps. Logarithmic vertical axes; exact-wing zero errors are omitted. Differences across caps use nested samples from one seed.


### Distribution Accuracy Versus Incremental Recovery Cost


In [ ]:
display(Image(filename=str(OUTPUT_DIR / "random_shell_cap_sensitivity.png")))


*Note.* Measured enclosing wall times for the nested cap ladder vs. central-shell KS and Wasserstein-1 error. The 100k cap achieves a 67% error reduction over 10k at ~2.89s runtime, providing an excellent engineering trade-off for practical exploration.


### Sample Fractions Under 1% Rule and Precision Caps


In [ ]:
display(Image(filename=str(OUTPUT_DIR / "random_shell_sample_fraction_by_k.png")))


*Note.* Fraction n/N of sampled models across shells. Capping binds in large central shells (k=11-19) without altering uniform sampling from undiscovered remainders.


## 9. Small-Wing Exact Refinement (Documented Next Design Refinement)

Under the 1% rule, boundary shells k=3 and k=27 contain only 41 random draws (N_3 = N_27 = 4,060), leaving higher relative variance at the tails.
An exact census for small wings:
- k <= 3 and k >= 27: costs only **9,052 models total**.
- k <= 4 and k >= 26: costs only **63,862 models total**.

This hybrid architecture - exact census for small shells + uniform random shell recovery for large shells + targeted BFG discovery for champions - is documented as the recommended next design refinement / experimental option.


## 10. Summary and Scientific Boundaries

1. **Targeted BFG Discovery** efficiently isolates top-performing models, MAP, and true causal structure in a small fraction of the space (B=5,000, runtime ~0.36 s).
2. **Stratified Random Shell Recovery** estimates reticular score distributions without equal-pooling bias (recommended cap 100k, runtime ~2.89 s).
3. **Exact Full Enumeration** remains the immutable gold standard for global posterior mass, exact PIPs, and exact model probabilities.
